## Part A: Support Vector Machine (SVM)

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

1. Load the dataset and perform the necessary preprocessing.

In [3]:
df = pd.read_csv("wdbc.data", header = None, names = ["id", "diagnosis"] + [f"feature_{i}" for i in range(1, 31)])

print(df.head())

         id diagnosis  feature_1  feature_2  feature_3  feature_4  feature_5  \
0    842302         M      17.99      10.38     122.80     1001.0    0.11840   
1    842517         M      20.57      17.77     132.90     1326.0    0.08474   
2  84300903         M      19.69      21.25     130.00     1203.0    0.10960   
3  84348301         M      11.42      20.38      77.58      386.1    0.14250   
4  84358402         M      20.29      14.34     135.10     1297.0    0.10030   

   feature_6  feature_7  feature_8  ...  feature_21  feature_22  feature_23  \
0    0.27760     0.3001    0.14710  ...       25.38       17.33      184.60   
1    0.07864     0.0869    0.07017  ...       24.99       23.41      158.80   
2    0.15990     0.1974    0.12790  ...       23.57       25.53      152.50   
3    0.28390     0.2414    0.10520  ...       14.91       26.50       98.87   
4    0.13280     0.1980    0.10430  ...       22.54       16.67      152.20   

   feature_24  feature_25  feature_26  featu

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          569 non-null    int64  
 1   diagnosis   569 non-null    str    
 2   feature_1   569 non-null    float64
 3   feature_2   569 non-null    float64
 4   feature_3   569 non-null    float64
 5   feature_4   569 non-null    float64
 6   feature_5   569 non-null    float64
 7   feature_6   569 non-null    float64
 8   feature_7   569 non-null    float64
 9   feature_8   569 non-null    float64
 10  feature_9   569 non-null    float64
 11  feature_10  569 non-null    float64
 12  feature_11  569 non-null    float64
 13  feature_12  569 non-null    float64
 14  feature_13  569 non-null    float64
 15  feature_14  569 non-null    float64
 16  feature_15  569 non-null    float64
 17  feature_16  569 non-null    float64
 18  feature_17  569 non-null    float64
 19  feature_18  569 non-null    float64
 20 

In [5]:
df.shape

(569, 32)

In [7]:
df.isnull().sum().sum()

np.int64(0)

In [9]:
df = df.drop("id", axis = 1)

# Encoding the target variable
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

X = df.drop("diagnosis", axis = 1)
y = df["diagnosis"]

print("Features:", X.shape)
print("Target:", y.shape)
print(y.value_counts())

Features: (569, 30)
Target: (569,)
diagnosis
0    357
1    212
Name: count, dtype: int64


2. Split the dataset into training and testing sets (80:20).

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42, stratify = y)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (455, 30)
Testing set: (114, 30)


3. Train an SVM classifier using a Linear Kernel and explore hyper parameter tuning.

In [11]:
pipeline = Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel = "linear"))])

param_grid = {"svm__C": [0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(pipeline, param_grid, cv = 5, scoring = "accuracy")

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

Best Parameters: {'svm__C': 0.1}
Best Cross-Validation Accuracy: 0.9692307692307693


In [12]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

4. Evaluate the model using:
- Accuracy
- Precision
- Recall
- F1 Score
- Confusion Matrix


In [13]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9824561403508771


In [14]:
precision = precision_score(y_test, y_pred)
print("Precision:", precision)

Precision: 1.0


In [15]:
recall = recall_score(y_test, y_pred)
print("Recall:", recall)

Recall: 0.9523809523809523


In [16]:
f1 = f1_score(y_test, y_pred)
print("F1 Score:", f1)

F1 Score: 0.975609756097561


In [17]:
print("Model Evaluation")
print(" ")
print(f"Accuracy :  {accuracy:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall   :  {recall:.4f}")
print(f"F1 Score :  {f1:.4f}")

Model Evaluation
 
Accuracy :  0.9825
Precision:  1.0000
Recall   :  0.9524
F1 Score :  0.9756


In [18]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[72  0]
 [ 2 40]]


5. Summarize your observations.

- The best hyperparameter was C = 0.1, with a cross-validation accuracy of 96.92%. This indicates that a relatively smaller value of C provided the best generalization performance for the Linear SVM.
- The Linear SVM achieved a test accuracy of 98.25%, meaning that 98.25% of the test samples were classified correctly.
- The precision was 100%, indicating that all samples predicted as malignant were actually malignant. There were no false positives.
- The recall was 95.24%, meaning that the model correctly identified 95.24% of the actual malignant cases. Out of 42 malignant cases, 40 were correctly identified and 2 were misclassified as benign.
- The F1 score of 97.56% indicates an excellent balance between precision and recall.

- True Negatives (TN) = 72
- False Positives (FP) = 0
- False Negatives (FN) = 2
- True Positives (TP) = 40

Overall, the Linear SVM performed very well on the Breast Cancer Wisconsin Diagnostic dataset. The high accuracy and F1 score demonstrate strong classification performance. However, the two false negatives are important because, in a medical diagnosis context, incorrectly classifying a malignant tumor as benign can be more serious than a false positive.